In [1]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


spark = SparkSession.builder \
    .appName("MinIO Verification") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.reporting", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.reporting.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.reporting.warehouse", "s3a://iceberg/iceberg/WideWorldImportersDW") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

In [2]:
print("--- Checking Namespaces ---")
spark.sql("SHOW NAMESPACES IN reporting").show()

print("--- Checking Tables in Dimension ---")
spark.sql("SHOW TABLES IN reporting.dimension").show()

--- Checking Namespaces ---
+---------+
|namespace|
+---------+
|dimension|
|     fact|
+---------+

--- Checking Tables in Dimension ---
+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|dimension|           dates|      false|
|dimension|  payment_method|      false|
|dimension|        supplier|      false|
|dimension|            city|      false|
|dimension|      stock_item|      false|
|dimension|        customer|      false|
|dimension|transaction_type|      false|
|dimension|        employee|      false|
+---------+----------------+-----------+



In [3]:
# 1. Check the total number of rows
print("--- Row Count for Customer Table ---")
spark.sql("SELECT COUNT(*) AS total_customers FROM reporting.dimension.customer").show()

# 2. Preview the first 5 rows to ensure the schema/columns look correct
print("--- Data Preview ---")
spark.sql("SELECT * FROM reporting.dimension.customer LIMIT 5").show(truncate=False)

--- Row Count for Customer Table ---
+---------------+
|total_customers|
+---------------+
|            403|
+---------------+

--- Data Preview ---
+------------+---------------+--------------------------------+---------------------------+------------+-------------+-----------------+-----------+-------------------+--------------------------+-----------+
|Customer_Key|WWI_Customer_ID|Customer                        |Bill_To_Customer           |Category    |Buying_Group |Primary_Contact  |Postal_Code|Valid_From         |Valid_To                  |Lineage_Key|
+------------+---------------+--------------------------------+---------------------------+------------+-------------+-----------------+-----------+-------------------+--------------------------+-----------+
|303         |502            |Wingtip Toys (Miesville, MN)    |Wingtip Toys (Head Office) |Novelty Shop|Wingtip Toys |Alejandro Escobar|90728      |2013-01-01 00:00:00|9999-12-31 23:59:59.999999|2          |
|355         |554  

In [5]:
from pyspark.sql import SparkSession

# 1. Spin up a session with BOTH catalogs defined
spark = SparkSession.builder \
    .appName("MinIO vs Local Verification") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    \
    .config("spark.sql.catalog.local_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.local_rpt.warehouse", "file:///C:/data/data_files/iceberg/WideWorldImportersDW") \
    \
    .config("spark.sql.catalog.minio_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.minio_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.minio_rpt.warehouse", "s3a://iceberg/iceberg/WideWorldImportersDW") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# 2. Run the comparison
print("Comparing row counts...")
local_count = spark.sql("SELECT COUNT(*) FROM local_rpt.dimension.customer").collect()[0][0]
minio_count = spark.sql("SELECT COUNT(*) FROM minio_rpt.dimension.customer").collect()[0][0]

if local_count == minio_count:
    print(f"Success! Both tables have exactly {local_count} rows.")
else:
    print(f"Mismatch! Local has {local_count}, MinIO has {minio_count}.")

Comparing row counts...
Success! Both tables have exactly 403 rows.
